<a href="https://colab.research.google.com/github/tfiglan/Geospatial-PowerAnalysis/blob/main/clarens_1csbo_ms3_test_sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
!pip install dss-python
import dss
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
import math

# --- Initial DSS Setup (outside the function, as these are static) ---
# In an offline environment like Spyder, you would typically manage the DSS-Python installation directly.
# If dss-python is not installed, run: !pip install dss-python

# NOTE: The 'added_path' logic for Colab might need adjustment if running outside Colab
# If running locally, ensure your DSS-Python environment is correctly set up.
# If the DSS engine requires a specific directory for circuit files, set it here.
# For this example, we assume dss-python can find its core components.

dss_engine = dss.DSS
DSSText = dss_engine.Text
DSSCircuit = dss_engine.ActiveCircuit
DSSSolution = dss_engine.ActiveCircuit.Solution
ControlQueue = dss_engine.ActiveCircuit.CtrlQueue
dss_engine.AllowForms = 0

# --- Global definitions for simulation loops ---
loading_scenarios_e2 = {
    'Peak': {'factor': 1.0, 'hours': 1750},
    'Medium': {'factor': 0.75, 'hours': 4380},
    'Minimum': {'factor': 0.25, 'hours': 2630}
}

original_peak_load_kW_e2 = {
    'Load_103': 16, 'Load_100': 16, 'Load_101': 16, 'Load_104': 16,
    'Load_99': 16, 'Load_102': 16, 'Load_111': 16, 'Load_112': 16,
    'Load_121': 16, 'Load_167': 16, 'Load_168': 16, 'Load_174': 16,
    'Load_175': 16, 'Load_176': 16, 'Load_161': 16, 'Load_162': 16,
    'Load_165': 16, 'Load_164': 16, 'Load_170': 16, 'Load_171': 16,
    'Load_154': 16, 'Load_151': 16, 'Load_150B': 16, 'Load_153': 16,
    'Load_152': 16, 'Load_49A': 16, 'Load_159': 16, 'Load_160': 16,
    'Load_154A': 16, 'Load_208': 16, 'Load_RE212': 16, 'Load_209A': 16,
    'Load_702': 16, 'Load_207': 16, 'Load_523A': 16, 'Load_523': 16,
    'Load_156': 25, 'Load_157': 25, 'Load_158': 25, 'Load_213': 25,
    'Load_163': 25, 'Load_169': 25, 'Load_128': 25, 'Load_158H': 25,
    'Load_158I': 25
}

house_buses_e2 = {
    'Load_103': 'C1', 'Load_100': 'C2', 'Load_101': 'F2', 'Load_104': 'F1',
    'Load_99': 'E2', 'Load_102': 'E1', 'Load_111': 'M1', 'Load_112': 'M2',
    'Load_121': 'HH1', 'Load_167': 'AA2', 'Load_168': 'AA1', 'Load_174': 'DD3',
    'Load_175': 'DD2', 'Load_176': 'DD1', 'Load_161': 'Z1', 'Load_162': 'Z2',
    'Load_165': 'FF1', 'Load_164': 'GG1', 'Load_170': 'GG2', 'Load_171': 'EE1',
    'Load_154': 'B1', 'Load_151': 'G1', 'Load_150B': 'H2', 'Load_153': 'H1',
    'Load_152': 'I1', 'Load_49A': 'I2', 'Load_159': 'P1', 'Load_160': 'P2',
    'Load_154A': 'S1', 'Load_208': 'T2', 'Load_RE212': 'W1', 'Load_209A': 'W2',
    'Load_702': 'W3', 'Load_207': 'U2', 'Load_523A': 'V1', 'Load_523': 'V2',
    'Load_156': 'N1', 'Load_157': 'N2', 'Load_158': 'N3', 'Load_213': 'Q1',
    'Load_163': 'BB2', 'Load_169': 'BB1', 'Load_128': 'CC1', 'Load_158H': 'L1',
    'Load_158I': 'L2'
}

nominal_voltage_pu = 1.0
lower_limit_pu = nominal_voltage_pu * 0.90
upper_limit_pu = nominal_voltage_pu * 1.10

# Define three-phase loads for kV value differentiation and voltage extraction logic
THREE_PHASE_LOADS = [
    'Load_156', 'Load_157', 'Load_158', 'Load_213', 'Load_163',
    'Load_169', 'Load_128', 'Load_158H', 'Load_158I'
]

# Store base PV system configurations (original pmpp for scaling)
PV_BASE_CONFIGS = {
    'PVCL30': {'bus': 'C1.1', 'phases': 1, 'kva': 10, 'pmpp': 10, 'kv': '(0.4 3 sqrt /)'},
    'PVCL60': {'bus': 'T2.1', 'phases': 1, 'kva': 30, 'pmpp': 30, 'kv': '(0.4 3 sqrt /)'},
    'PVCL61': {'bus': 'W2.1', 'phases': 1, 'kva': 5, 'pmpp': 5, 'kv': '(0.4 3 sqrt /)'},
    'PVCL57': {'bus': 'Z1.1', 'phases': 1, 'kva': 5, 'pmpp': 5, 'kv': '(0.4 3 sqrt /)'},
    'PVCL58': {'bus': 'Z2.1', 'phases': 1, 'kva': 10, 'pmpp': 10, 'kv': '(0.4 3 sqrt /)'},
    'PVCL50': {'bus': 'FF1.1', 'phases': 1, 'kva': 1, 'pmpp': 1, 'kv': '(0.4 3 sqrt /)'},
    'PVCL36': {'bus': 'DD3.1', 'phases': 1, 'kva': 3, 'pmpp': 3, 'kv': '(0.4 3 sqrt /)'},
    'PVCL52': {'bus': 'DD2.1', 'phases': 1, 'kva': 5, 'pmpp': 5, 'kv': '(0.4 3 sqrt /)'},
    'PVCL59': {'bus': 'BB2.1.2.3', 'phases': 3, 'kva': 3, 'pmpp': 3, 'kv': '0.4'},
    'PVCL56': {'bus': 'N2.1.2.3', 'phases': 3, 'kva': 5, 'pmpp': 5, 'kv': '0.4'},
    'PVCL104': {'bus': 'L1.1.2.3', 'phases': 3, 'kva': 30, 'pmpp': 30, 'kv': '0.4'},
    'PVCL37': {'bus': 'L2.1.2.3', 'phases': 3, 'kva': 3, 'pmpp': 3, 'kv': '0.4'}
}

# Helper function to process voltage DataFrames for plotting
def process_voltage_df_for_plotting(df, pv_status_label):
    processed_data = []
    bus_columns = [col for col in df.columns if col not in ['Scenario', 'Compliant']]

    for _, row in df.iterrows():
        scenario = row['Scenario']
        for col_name in bus_columns:
            bus_base = col_name.replace(' (pu)', '') # Extract bus name without '(pu)'
            voltage_value = row[col_name]

            if isinstance(voltage_value, str) and 'P1:' in voltage_value: # Three-phase bus
                phases = voltage_value.split(', ')
                for phase_entry in phases:
                    parts = phase_entry.split(':')
                    phase_name = parts[0]
                    voltage_pu = float(parts[1])
                    processed_data.append({
                        'Scenario': scenario,
                        'Bus': bus_base,
                        'Phase': phase_name,
                        'Voltage (pu)': voltage_pu,
                        'PV Status': pv_status_label
                    })
            else: # Single-phase bus
                voltage_pu = float(voltage_value)
                processed_data.append({
                    'Scenario': scenario,
                    'Bus': bus_base,
                    'Phase': 'All',
                    'Voltage (pu)': voltage_pu,
                    'PV Status': pv_status_label
                })
    return pd.DataFrame(processed_data)


def run_simulation_and_analysis(global_load_multiplier, new_power_factor, global_pv_multiplier):
    """
    Encapsulates the core OpenDSS simulation and analysis logic.
    Accepts global multipliers for load and PV, and a new power factor for loads.
    Returns annual losses and voltage compliance dataframes for scenarios with and without PV,
    and net demand components.
    """
    global dss_engine, DSSText, DSSCircuit, DSSSolution, ControlQueue

    DSSText.Command = 'Clear'

    # --- 2. Network Modelling - Creating a Circuit ---
    DSSText.Command = 'Set DefaultBaseFrequency=50'
    DSSText.Command = 'New Circuit.Clarens_Test_LV_Network'
    DSSText.Command = 'Edit vsource.source bus1=sourceBus basekv=22 pu=1.0 phases=3'

    # --- 3. Adding the 22kV/0.4kV Transformer ---
    DSSText.Command = 'New transformer.1CSBO-MS3 Buses=[sourcebus, A.1.2.3] Conns=[delta wye] KVs=[22, 0.4] KVAs=[315 315] %Rs=0.00 xhl=2.5 %loadloss=0 '

    # --- 4. Creating the linecodes ---
    DSSText.Command = 'new linecode.240sq nphases=3 R1=0.127 X1=0.072 R0=0.342 X0=0.089 units=km'
    DSSText.Command = 'new linecode.16sq nphases=1 R1=1.15 X1=0.083 R0=1.2 X0=0.083 units=km'
    DSSText.Command = 'new linecode.185sq nphases=3 R1=0.078 X1=0.042 R0=0.108 X0=0.093 units=km'

    # --- 5. Adding the 400V and 230V lines ---
    # Three-phase lines
    DSSText.Command = 'new line.A_B bus1=A.1.2.3 bus2=B.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.A_C bus1=A.1.2.3 bus2=C.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.A_D bus1=A.1.2.3 bus2=D.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.D_J bus1=D.1.2.3 bus2=J.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.J_KK bus1=J.1.2.3 bus2=KK.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.KK_CC bus1=KK.1.2.3 bus2=CC.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.B_G bus1=B.1.2.3 bus2=G.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.G_H bus1=G.1.2.3 bus2=H.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.H_I bus1=H.1.2.3 bus2=I.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.C_F bus1=C.1.2.3 bus2=F.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.C_E bus1=C.1.2.3 bus2=E.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.D_N bus1=D.1.2.3 bus2=N.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.N_N1 bus1=N.1.2.3 bus2=N1.1.2.3 length=0.015 phases=3 units=km linecode=185sq'
    DSSText.Command = 'new line.N_N2 bus1=N.1.2.3 bus2=N2.1.2.3 length=0.015 phases=3 units=km linecode=185sq'
    DSSText.Command = 'new line.N_N3 bus1=N.1.2.3 bus2=N3.1.2.3 length=0.015 phases=3 units=km linecode=185sq'
    DSSText.Command = 'new line.N_O bus1=N.1.2.3 bus2=O.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.O_P bus1=O.1.2.3 bus2=P.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.O_Q bus1=O.1.2.3 bus2=Q.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.Q_Q1 bus1=Q.1.2.3 bus2=Q1.1.2.3 length=0.015 phases=3 units=km linecode=185sq'
    DSSText.Command = 'new line.Q_R bus1=Q.1.2.3 bus2=R.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.R_S bus1=R.1.2.3 bus2=S.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.R_W bus1=R.1.2.3 bus2=W.1.2.3 length=0.10 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.R_T bus1=R.1.2.3 bus2=T.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.T_T1 bus1=T.1.2.3 bus2=T1.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.T_U bus1=T.1.2.3 bus2=U.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.U_U1 bus1=U.1.2.3 bus2=U1.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.U_V bus1=U.1.2.3 bus2=V.1.2.3 length=0.3 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.J_L bus1=J.1.2.3 bus2=L.1.2.3 length=0.1 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.L_M bus1=L.1.2.3 bus2=M.1.2.3 length=0.1 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.KK_HH bus1=KK.1.2.3 bus2=HH.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.KK_AA bus1=KK.1.2.3 bus2=AA.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.CC_CC1 bus1=CC.1.2.3 bus2=CC1.1.2.3 length=0.015 phases=3 units=km linecode=185sq'
    DSSText.Command = 'new line.CC_DD bus1=CC.1.2.3 bus2=DD.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.J_Y bus1=J.1.2.3 bus2=Y.1.2.3 length=0.1 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.Y_Z bus1=Y.1.2.3 bus2=Z.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.Y_BB bus1=Y.1.2.3 bus2=BB.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.BB_BB1 bus1=BB.1.2.3 bus2=BB1.1.2.3 length=0.015 phases=3 units=km linecode=185sq'
    DSSText.Command = 'new line.BB_BB2 bus1=BB.1.2.3 bus2=BB2.1.2.3 length=0.015 phases=3 units=km linecode=185sq'
    DSSText.Command = 'new line.BB_EE bus1=BB.1.2.3 bus2=EE.1.2.3 length=0.1 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.EE_FF bus1=EE.1.2.3 bus2=FF.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.FF_GG bus1=FF.1.2.3 bus2=GG.1.2.3 length=0.05 phases=3 units=km linecode=240sq'
    DSSText.Command = 'new line.L_L1 bus1=L.1.2.3 bus2=L1.1.2.3 length=0.015 phases=3 units=km linecode=185sq'
    DSSText.Command = 'new line.L_L2 bus1=L.1.2.3 bus2=L2.1.2.3 length=0.015 phases=3 units=km linecode=185sq'

    # Single-phase lines
    DSSText.Command = 'new line.B_B1 bus1=B.1 bus2=B1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.G_G1 bus1=G.1 bus2=G1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.H_H1 bus1=H.1 bus2=H1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.H_H2 bus1=H.2 bus2=H2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.I_I1 bus1=I.1 bus2=I1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.I_I2 bus1=I.2 bus2=I2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.C_C1 bus1=C.1 bus2=C1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.C_C2 bus1=C.2 bus2=C2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.F_F1 bus1=F.1 bus2=F1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.F_F2 bus1=F.2 bus2=F2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.E_E1 bus1=E.1 bus2=E1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.E_E2 bus1=E.2 bus2=E2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.M_M1 bus1=M.1 bus2=M1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.M_M2 bus1=M.2 bus2=M2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.P_P1 bus1=P.1 bus2=P1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.P_P2 bus1=P.2 bus2=P2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.W_W1 bus1=W.1 bus2=W1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.W_W2 bus1=W.2 bus2=W2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.W_W3 bus1=W.3 bus2=W3.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.S_S1 bus1=S.1 bus2=S1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.T1_T2 bus1=T1.1 bus2=T2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.U1_U2 bus1=U1.1 bus2=U2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.V_V1 bus1=V.1 bus2=V1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.V_V2 bus1=V.2 bus2=V2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.Z_Z1 bus1=Z.1 bus2=Z1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.Z_Z2 bus1=Z.2 bus2=Z2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.EE_EE1 bus1=EE.1 bus2=EE1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.FF_FF1 bus1=FF.1 bus2=FF1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.GG_GG1 bus1=GG.1 bus2=GG1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.GG_GG2 bus1=GG.2 bus2=GG2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.HH_HH1 bus1=HH.1 bus2=HH1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.AA_AA1 bus1=AA.1 bus2=AA1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.AA_AA2 bus1=AA.2 bus2=AA2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.DD_DD1 bus1=DD.1 bus2=DD1.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.DD_DD2 bus1=DD.2 bus2=DD2.1 length=0.015 phases=1 units=km linecode=16sq'
    DSSText.Command = 'new line.DD_DD3 bus1=DD.3 bus2=DD3.1 length=0.015 phases=1 units=km linecode=16sq'

    # --- 6. Add Base Loads ---
    original_power_factor = 0.95  # Defined in the original notebook
    for load_name, peak_kw in original_peak_load_kW_e2.items():
        if load_name in THREE_PHASE_LOADS:
            DSSText.Command = f'new load.{load_name} bus1={house_buses_e2[load_name]}.1.2.3 phases=3 kV=0.4 kW={peak_kw} pf={original_power_factor} model=1 conn=wye Vminpu=0.85 Vmaxpu=1.20 status=fixed'
        else:  # single-phase loads
            DSSText.Command = f'new load.{load_name} bus1={house_buses_e2[load_name]}.1 phases=1 kV=(0.4 3 sqrt /) kW={peak_kw} pf={original_power_factor} model=1 conn=wye Vminpu=0.85 Vmaxpu=1.20 status=fixed'

    # --- 10. Add Base PV Systems ---
    for pv_name, config in PV_BASE_CONFIGS.items():
        DSSText.Command = f"new PVSystem.{pv_name} bus1={config['bus']} phases={config['phases']} kva={config['kva']} pmpp={config['pmpp']} pf=1 kV={config['kv']} model=1"

    # --- 7. Set the Control mode and the Voltage bases ---
    DSSText.Command = 'set controlmode=static'
    DSSText.Command = 'set mode=snapshot'
    DSSText.Command = 'Set VoltageBases=[22 0.4]'
    DSSText.Command = 'calcvoltagebases'
    DSSText.Command = 'Set maxcontroliter=1000'

    # --- Results storage ---
    total_annual_losses_kwh_e2 = 0.0
    all_results_e2_voltages = []
    total_annual_losses_kwh_e2_pv = 0.0
    all_results_e2_voltages_pv = []

    # Data for net demand calculation
    all_scenario_load_data = []  # To calculate total load kW per scenario
    total_pv_generation_per_scenario = []

    # --- Simulation WITHOUT PV ---
    DSSText.Command = 'Disable PVSystem.*'  # Ensure all PVs are disabled

    for scenario_name, data in loading_scenarios_e2.items():
        current_load_factor = data['factor'] * global_load_multiplier
        scenario_hours = data['hours']
        scenario_total_load_kw = 0.0
        scenario_total_load_kvar = 0.0

        # Apply loading factor and new power factor to each load
        for load_name, peak_kw in original_peak_load_kW_e2.items():
            current_kw = peak_kw * current_load_factor
            # Ensure the argument for acos is within [-1, 1]
            pf_clamped = max(-1.0, min(1.0, new_power_factor))
            current_kvar = current_kw * math.tan(math.acos(pf_clamped)) if pf_clamped != 1 else 0.0
            DSSText.Command = f'Edit load.{load_name} kW={current_kw} pf={new_power_factor}'
            scenario_total_load_kw += current_kw
            scenario_total_load_kvar += current_kvar

        DSSSolution.Solve()
        if not DSSSolution.Converged:
            print(f"Warning: Circuit did not converge for {scenario_name} scenario (without PV).")
            continue

        # E2.1: Calculate energy losses
        line_loss = DSSCircuit.LineLosses
        active_power_losses = abs(line_loss[0]) / 1000  # Convert W to kW
        total_annual_losses_kwh_e2 += active_power_losses * scenario_hours

        # E2.2: Extract voltages and check compliance
        scenario_voltages = {'Scenario': scenario_name}
        overall_compliant = True
        for load_name, bus_base in house_buses_e2.items():
            if load_name in THREE_PHASE_LOADS:
                DSSCircuit.SetActiveBus(bus_base)
                phase_voltages = []
                compliance_issue = False
                for phase_idx in range(1, 4):
                    voltage_pu = round(DSSCircuit.ActiveBus.puVmagAngle[2 * phase_idx - 2], 3)
                    phase_voltages.append(voltage_pu)
                    if not (lower_limit_pu <= voltage_pu <= upper_limit_pu):
                        compliance_issue = True
                scenario_voltages[f'{bus_base} (pu)'] = f"P1:{phase_voltages[0]}, P2:{phase_voltages[1]}, P3:{phase_voltages[2]}"
                if compliance_issue:
                    overall_compliant = False
            else:
                bus_full_name = f'{bus_base}.1'
                DSSCircuit.SetActiveBus(bus_full_name)
                voltage_pu = round(DSSCircuit.ActiveBus.puVmagAngle[0], 3)
                scenario_voltages[f'{bus_base} (pu)'] = voltage_pu
                if not (lower_limit_pu <= voltage_pu <= upper_limit_pu):
                    overall_compliant = False
        scenario_voltages['Compliant'] = 'Yes' if overall_compliant else 'No'
        all_results_e2_voltages.append(scenario_voltages)

        # Store total load data for net demand calculation
        scenario_total_load_kVA = math.sqrt(scenario_total_load_kw**2 + scenario_total_load_kvar**2)
        all_scenario_load_data.append({
            'Scenario': scenario_name,
            'Total Active Power (kW)': round(scenario_total_load_kw, 2),
            'Total Reactive Power (kVAr)': round(scenario_total_load_kvar, 2),
            'Total Apparent Power (kVA)': round(scenario_total_load_kVA, 2)
        })

    df_voltages_e2 = pd.DataFrame(all_results_e2_voltages)
    df_total_load_demand = pd.DataFrame(all_scenario_load_data)

    # --- Simulation WITH PV ---
    DSSText.Command = 'Enable PVSystem.*'  # Enable all PVs

    for scenario_name, data in loading_scenarios_e2.items():
        current_load_factor = data['factor'] * global_load_multiplier
        scenario_hours = data['hours']
        scenario_total_pv_gen_kw = 0.0

        # Apply loading factor and new power factor to each load (same as without PV)
        for load_name, peak_kw in original_peak_load_kW_e2.items():
            current_kw = peak_kw * current_load_factor
            pf_clamped = max(-1.0, min(1.0, new_power_factor))
            current_kvar = current_kw * math.tan(math.acos(pf_clamped)) if pf_clamped != 1 else 0.0
            DSSText.Command = f'Edit load.{load_name} kW={current_kw} pf={new_power_factor}'

        # Apply global_pv_multiplier to PV systems
        for pv_name, config in PV_BASE_CONFIGS.items():
            current_pmpp_pv = config['pmpp'] * global_pv_multiplier
            DSSText.Command = f"Edit PVSystem.{pv_name} pmpp={current_pmpp_pv}"
            scenario_total_pv_gen_kw += current_pmpp_pv

        DSSSolution.Solve()
        if not DSSSolution.Converged:
            print(f"Warning: Circuit with PV did not converge for {scenario_name} scenario.")
            continue

        # E2.3: Calculate energy losses
        line_loss = DSSCircuit.LineLosses
        active_power_losses = abs(line_loss[0]) / 1000
        total_annual_losses_kwh_e2_pv += active_power_losses * scenario_hours

        # E2.4: Extract voltages and check compliance
        scenario_voltages_pv = {'Scenario': scenario_name}
        overall_compliant_pv = True
        for load_name, bus_base in house_buses_e2.items():
            if load_name in THREE_PHASE_LOADS:
                DSSCircuit.SetActiveBus(bus_base)
                phase_voltages = []
                compliance_issue = False
                for phase_idx in range(1, 4):
                    voltage_pu = round(DSSCircuit.ActiveBus.puVmagAngle[2 * phase_idx - 2], 3)
                    phase_voltages.append(voltage_pu)
                    if not (lower_limit_pu <= voltage_pu <= upper_limit_pu):
                        compliance_issue = True
                scenario_voltages_pv[f'{bus_base} (pu)'] = f"P1:{phase_voltages[0]}, P2:{phase_voltages[1]}, P3:{phase_voltages[2]}"
                if compliance_issue:
                    overall_compliant_pv = False
            else:
                bus_full_name = f'{bus_base}.1'
                DSSCircuit.SetActiveBus(bus_full_name)
                voltage_pu = round(DSSCircuit.ActiveBus.puVmagAngle[0], 3)
                scenario_voltages_pv[f'{bus_base} (pu)'] = voltage_pu
                if not (lower_limit_pu <= voltage_pu <= upper_limit_pu):
                    overall_compliant_pv = False
        scenario_voltages_pv['Compliant'] = 'Yes' if overall_compliant_pv else 'No'
        all_results_e2_voltages_pv.append(scenario_voltages_pv)

        # Store total PV generation for net demand calculation
        total_pv_generation_per_scenario.append({
            'Scenario': scenario_name,
            'Total PV Generation (kW)': round(scenario_total_pv_gen_kw, 2)
        })

    df_voltages_e2_pv = pd.DataFrame(all_results_e2_voltages_pv)
    df_total_pv_generation = pd.DataFrame(total_pv_generation_per_scenario)

    # --- Calculate Net Demand Components ---
    df_net_demand_components = pd.merge(
        df_total_load_demand,
        df_total_pv_generation,
        on='Scenario',
        how='left'
    )
    df_net_demand_components['Net Active Power (kW)'] = (
        df_net_demand_components['Total Active Power (kW)']
        - df_net_demand_components['Total PV Generation (kW)']
    )

    return (
        total_annual_losses_kwh_e2, df_voltages_e2,
        total_annual_losses_kwh_e2_pv, df_voltages_e2_pv,
        df_net_demand_components
    )

def display_dashboard_results(global_load_multiplier, new_power_factor, global_pv_multiplier):
    clear_output(wait=True)

    # 1. Run simulation and analysis with current parameters
    (losses_no_pv, df_voltages_no_pv,
     losses_with_pv, df_voltages_with_pv,
     df_net_demand_components) = run_simulation_and_analysis(
        global_load_multiplier, new_power_factor, global_pv_multiplier
    )

    print("### Simulation Results Overview\n")

    # 2. Display Annual Energy Losses
    print(f"*   **Annual Energy Losses (No PV):** {losses_no_pv:.2f} kWh")
    print(f"*   **Annual Energy Losses (With PV):** {losses_with_pv:.2f} kWh")
    print(f"*   **Difference in Annual Energy Losses (No PV - With PV):** {(losses_no_pv - losses_with_pv):.2f} kWh\n")

    # 3. Process voltage DataFrames for plotting
    df_no_pv_long = process_voltage_df_for_plotting(df_voltages_no_pv, 'No PV')
    df_with_pv_long = process_voltage_df_for_plotting(df_voltages_with_pv, 'With PV')
    df_all_voltages_long = pd.concat([df_no_pv_long, df_with_pv_long], ignore_index=True)

    # Combine 'Bus' and 'Phase' for a clearer x-axis if there are multi-phase buses
    df_all_voltages_long['Bus_Phase'] = df_all_voltages_long.apply(
        lambda row: f"{row['Bus']}-{row['Phase']}" if row['Phase'] != 'All' else row['Bus'],
        axis=1
    )

    # Order the scenarios for consistent plotting
    scenario_order = ['Minimum', 'Medium', 'Peak']
    df_all_voltages_long['Scenario'] = pd.Categorical(df_all_voltages_long['Scenario'], categories=scenario_order, ordered=True)

    # Sort by Bus_Phase to ensure consistent x-axis order across facets
    df_all_voltages_long = df_all_voltages_long.sort_values(by=['Bus_Phase', 'Scenario']).reset_index(drop=True)


    # 4. Create the voltage compliance plot
    num_buses = len(df_all_voltages_long['Bus_Phase'].unique())
    fig_height = 6
    fig_aspect = num_buses / 40

    g = sns.relplot(
        data=df_all_voltages_long,
        x='Bus_Phase',
        y='Voltage (pu)',
        hue='Scenario',
        col='PV Status',
        col_wrap=1, # Display 'No PV' and 'With PV' as separate columns
        kind='line',
        marker='o',
        height=fig_height,
        aspect=fig_aspect,
        facet_kws={'sharex': False, 'sharey': True},
        palette={'Peak': 'red', 'Medium': 'orange', 'Minimum': 'green'},
        legend=False # Disable default legend from relplot
    )

    for ax in g.axes.flat:
        ax.axhline(y=upper_limit_pu, color='gray', linestyle='--', label=f'Upper Limit ({upper_limit_pu:.2f} pu)')
        ax.axhline(y=lower_limit_pu, color='gray', linestyle='--', label=f'Lower Limit ({lower_limit_pu:.2f} pu)')
        ax.tick_params(axis='x', rotation=90)
        ax.set_xlabel('Bus and Phase', fontsize=10)
        ax.set_ylim(min(lower_limit_pu - 0.05, df_all_voltages_long['Voltage (pu)'].min() - 0.02),
                    max(upper_limit_pu + 0.05, df_all_voltages_long['Voltage (pu)'].max() + 0.02))

    g.set_axis_labels("Bus and Phase", "Voltage (pu)")
    g.set_titles("PV Status: {col_name}")
    plt.suptitle('Voltage Compliance Across Scenarios with and without PV', y=1.02, fontsize=16)

    # Explicitly create handles and labels for Scenario (hue)
    scenario_handles = [plt.Line2D([0], [0], color='red', lw=2),
                        plt.Line2D([0], [0], color='orange', lw=2),
                        plt.Line2D([0], [0], color='green', lw=2)]
    scenario_labels = ['Peak', 'Medium', 'Minimum']

    # Explicitly create handles and labels for Voltage Limits
    limit_handles = [plt.Line2D([0], [0], color='gray', linestyle='--', lw=2),
                     plt.Line2D([0], [0], color='gray', linestyle='--', lw=2)]
    limit_labels = [f'Upper Limit ({upper_limit_pu:.2f} pu)',
                    f'Lower Limit ({lower_limit_pu:.2f} pu)']

    # Create the Scenario legend
    g.fig.legend(handles=scenario_handles, labels=scenario_labels, title='Scenario', bbox_to_anchor=(1.02, 0.9), loc='upper left')

    # Create the Voltage Limits legend
    g.fig.legend(handles=limit_handles, labels=limit_labels, title='Voltage Limits', bbox_to_anchor=(1.02, 0.7), loc='upper left')

    plt.tight_layout(rect=[0, 0, 0.9, 1]) # Adjust rect to make space for two legends
    plt.show()

    # 5. Display Net Demand Components
    print("### Net Demand Components\n")
    # Using to_string() for potentially tighter column spacing
    with pd.option_context('display.max_colwidth', None):
        print(df_net_demand_components.to_string(index=False))

# Create ipywidgets for input parameters
global_load_multiplier_widget = widgets.FloatSlider(
    value=1.0,
    min=0.5,
    max=2.0,
    step=0.1,
    description='Global Load Multiplier:',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

new_power_factor_widget = widgets.FloatSlider(
    value=0.95,
    min=0.8,
    max=1.0,
    step=0.01,
    description='New Power Factor:',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.2f',
)

global_pv_multiplier_widget = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=3.0,
    step=0.1,
    description='Global PV Multiplier:',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

from ipywidgets import interactive

# Create the interactive dashboard
interactive_dashboard = interactive(
    display_dashboard_results,
    global_load_multiplier=global_load_multiplier_widget,
    new_power_factor=new_power_factor_widget,
    global_pv_multiplier=global_pv_multiplier_widget
)

# Display the interactive dashboard
display(interactive_dashboard)

interactive(children=(FloatSlider(value=1.0, continuous_update=False, description='Global Load Multiplier:', m…